# 05 — KPI Export & Power BI Validation

This notebook loads the 17 commercial KPIs from `src.kpi_builder`, reviews the
what-if scenario table, and confirms all 7 Power BI export CSVs are present and
non-empty.  These files are the final handoff to Power BI Desktop.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import pathlib
import warnings
warnings.filterwarnings('ignore')
print('Ready')

In [ ]:
EXPORTS = pathlib.Path('..') / 'data' / 'powerbi_exports'

# 17 commercial KPIs
kpis = pd.read_csv(EXPORTS / 'commercial_kpis.csv')
print(f'KPI rows: {len(kpis)}')
print()
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
print(kpis.to_string(index=False))

In [ ]:
# What-if scenarios sample
wi = pd.read_csv(EXPORTS / 'what_if_scenarios.csv')
print(f'What-if scenario rows: {len(wi)}')
print(f'Beverage categories  : {wi["beverage_category"].nunique()}')
print(f'Discount levels      : {sorted(wi["discount_pct"].unique().tolist())}')
print(f'Coupon applied (Y/N) : {wi["coupon_applied"].unique().tolist()}')
print()
print('Sample rows (Carbonated Soft Drinks):')
print(wi[wi['beverage_category'] == 'Carbonated Soft Drinks'].head(6).to_string(index=False))

In [ ]:
# Validate all 7 Power BI CSVs
expected = [
    'fact_transactions_beverage.csv',
    'dim_products_beverage.csv',
    'dim_households.csv',
    'promotion_performance.csv',
    'household_segments.csv',
    'commercial_kpis.csv',
    'what_if_scenarios.csv',
]

print('Power BI export validation:')
print(f'{"File":<40} {"Exists":<8} {"Rows":>8} {"Size (KB)":>10}')
print('-' * 70)
all_ok = True
for fname in expected:
    fpath = EXPORTS / fname
    exists = fpath.exists()
    if exists:
        df = pd.read_csv(fpath, nrows=None if fname != 'fact_transactions_beverage.csv' else None)
        nrows = len(df)
        size_kb = round(fpath.stat().st_size / 1024, 1)
        print(f'{fname:<40} {"YES":<8} {nrows:>8,} {size_kb:>9.1f}')
    else:
        print(f'{fname:<40} {"MISSING":<8}')
        all_ok = False

print()
print('All files present:', all_ok)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pathlib

CHARTS = pathlib.Path('..') / 'outputs' / 'charts'

# Simple KPI category breakdown chart
kpi_by_cat = kpis.groupby('kpi_category').size()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(kpi_by_cat.index, kpi_by_cat.values, color='#17becf', edgecolor='white')
for i, val in enumerate(kpi_by_cat.values):
    ax.text(i, val + 0.05, str(val), ha='center', va='bottom', fontsize=10)
ax.set_title('Commercial KPIs by Category (17 total)', fontsize=11)
ax.set_ylabel('Number of KPIs', fontsize=10)
ax.set_xticks(range(len(kpi_by_cat)))
ax.set_xticklabels(kpi_by_cat.index, rotation=10, ha='right', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, kpi_by_cat.max() + 1)
plt.tight_layout()
fig.savefig(CHARTS / 'commercial_kpis_overview.png', dpi=130)
plt.close(fig)
print('Saved commercial_kpis_overview.png')

## Summary — KPI Export

All **7 Power BI export CSVs** are present and validated:

| File | Rows |
|---|---|
| fact_transactions_beverage.csv | 235,979 |
| dim_products_beverage.csv | 4,910 |
| dim_households.csv | 2,487 |
| promotion_performance.csv | 4,897 |
| household_segments.csv | 4 |
| commercial_kpis.csv | 17 |
| what_if_scenarios.csv | 70 |

**17 commercial KPIs** across 4 categories: Revenue, Mix, Promo, Customer.

**70 what-if scenarios**: 7 beverage categories x 5 discount levels (0/5/10/15/20%) x 2 coupon states (Y/N).
All scenario rows carry the label: *'Directional scenario only — not causal'*.
These CSVs are ready for import into Power BI Desktop.